# Models

[LLM](https://en.wikipedia.org/wiki/Large_language_model) là các công cụ AI mạnh mẽ có khả năng diễn giải và tạo văn bản giống con người. Chúng có tính linh hoạt cao, có thể viết nội dung, dịch ngôn ngữ, tóm tắt và trả lời câu hỏi mà không cần huấn luyện chuyên biệt cho từng nhiệm vụ.

Bên cạnh việc tạo văn bản, nhiều model còn hỗ trợ:

* Tool calling - gọi các tool bên ngoài (như truy vấn database hoặc gọi API) và sử dụng kết quả trong phản hồi của mình.
* Structured output - phản hồi của model được ràng buộc phải theo một format đã định nghĩa sẵn.
* Multimodality - xử lý và trả về dữ liệu không phải văn bản, chẳng hạn như hình ảnh, âm thanh và video.
* Reasoning - model thực hiện reasoning nhiều bước để đi đến kết luận.

Model là bộ máy reasoning của các agent. Chúng điều khiển quá trình ra quyết định của agent, xác định tool nào cần gọi, cách diễn giải kết quả và khi nào đưa ra câu trả lời cuối cùng.

Chất lượng và khả năng của model bạn chọn ảnh hưởng trực tiếp đến độ tin cậy và hiệu suất cơ bản của agent. Các model khác nhau sẽ mạnh ở các nhiệm vụ khác nhau - một số giỏi tuân theo các chỉ dẫn phức tạp, số khác giỏi reasoning có cấu trúc, và một số hỗ trợ context window lớn hơn để xử lý nhiều thông tin hơn.

Các standard model interface của LangChain cho phép bạn truy cập nhiều tích hợp từ các provider khác nhau, giúp bạn dễ dàng thử nghiệm và chuyển đổi giữa các model để tìm ra lựa chọn phù hợp nhất cho use case của mình.

## Cách sử dụng cơ bản

Model có thể được sử dụng theo hai cách:

1. **Cùng với agent** - Model có thể được chỉ định động khi tạo một agent.
2. **Độc lập** - Model có thể được gọi trực tiếp (ngoài agent loop) cho các nhiệm vụ như tạo văn bản, phân loại hoặc trích xuất dữ liệu mà không cần agent framework.

Cùng một model interface hoạt động trong cả hai trường hợp, cho bạn sự linh hoạt để bắt đầu đơn giản và mở rộng dần lên các workflow dựa trên agent phức tạp hơn khi cần.

### Khởi tạo một model

Cách dễ nhất để bắt đầu với một model độc lập trong LangChain là sử dụng `init_chat_model` để khởi tạo model từ provider chat model bạn chọn (các ví dụ dưới đây):

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.7-flash")

### Các provider và model được hỗ trợ

LangChain hỗ trợ tất cả các provider model lớn thông qua các package tích hợp chuyên dụng. Mỗi package provider triển khai cùng một standard interface, vì vậy bạn có thể hoán đổi provider mà không cần viết lại logic ứng dụng. Các model mới hoạt động ngay lập tức - không cần cập nhật LangChain - vì các package provider truyền tên model trực tiếp đến API của provider.

### Các method chính

- Invoke: Model nhận message làm input và trả về message sau khi tạo xong toàn bộ phản hồi.
- Stream: Gọi model, nhưng stream output ngay khi nó được tạo ra theo thời gian thực.
- Batch: Gửi nhiều request đến model theo batch để xử lý hiệu quả hơn.

## Parameters

Một chat model nhận các parameter dùng để cấu hình hành vi của nó. Toàn bộ các parameter được hỗ trợ tùy thuộc vào từng model và provider, nhưng các parameter chuẩn bao gồm:

- `model`: Tên hoặc định danh của model cụ thể bạn muốn sử dụng với một provider. Bạn cũng có thể chỉ định cả model và provider của nó trong một argument duy nhất theo format '{model_provider}:{model}', ví dụ 'openai:o1'.
- `api_key`: Key cần thiết để xác thực với provider của model. Key này thường được cấp khi bạn đăng ký quyền truy cập model. Thường được truy cập bằng cách thiết lập một biến môi trường.
- `temperature`: Kiểm soát mức độ ngẫu nhiên trong output của model. Giá trị càng cao khiến phản hồi càng sáng tạo; giá trị càng thấp khiến phản hồi càng xác định.
- `max_tokens`: Giới hạn tổng số token trong phản hồi, qua đó kiểm soát độ dài tối đa của output.
- `timeout`: Thời gian tối đa (tính bằng giây) để chờ phản hồi từ model trước khi hủy request.
- `max_retries`: Số lần thử tối đa mà hệ thống sẽ thực hiện để gửi lại request nếu nó thất bại do các vấn đề như network timeout hoặc rate limit. Việc retry sử dụng exponential backoff kèm jitter. Lỗi network, rate limit (429) và lỗi server (5xx) sẽ được tự động retry. Lỗi client như 401 (unauthorized) hoặc 404 sẽ không được retry. Đối với các nhiệm vụ agent chạy trong thời gian dài trên network không ổn định, hãy xem xét tăng giá trị này lên 10-15.

Khi sử dụng `init_chat_model`, hãy truyền các parameter này dưới dạng `**kwargs` trực tiếp:

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "google_genai:gemini-3.7-flash",
    # Kwargs được truyền vào model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # Giá trị mặc định; tăng lên nếu network không ổn định
)

### Khả năng phục hồi kết nối

Các chat model của LangChain tự động retry các API request thất bại bằng exponential backoff. Theo mặc định, model sẽ retry tối đa **6 lần** đối với lỗi network, rate limit (429) và lỗi server (5xx). Lỗi client như 401 (unauthorized) hoặc 404 sẽ không được retry.

Bạn có thể điều chỉnh `max_retries` và `timeout` khi tạo model, sau đó truyền instance đó vào `create_agent`, `create_deep_agent`, hoặc gọi độc lập:

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "google_genai:gemini-3.6-flash",
    max_retries=10,  # Tăng lên nếu network không ổn định (mặc định: 6)
    timeout=120,  # Đơn vị giây; tăng lên nếu kết nối chậm
)

Đối với các agent graph chạy trong thời gian dài trên network không ổn định, hãy xem xét tăng `max_retries` (ví dụ 10-15) và sử dụng `checkpointer` để tiến trình được lưu lại qua các lần thất bại.

## Invocation

Một chat model phải được gọi (invoke) để tạo ra output. Có ba phương thức gọi chính, mỗi phương thức phù hợp với các use case khác nhau.

### Invoke

Cách gọi model đơn giản nhất là sử dụng `invoke()` với một message đơn hoặc một danh sách các message.

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

response = model.invoke("Tại sao vẹt lại có bộ lông sặc sỡ?")
print(response)

content=[{'type': 'text', 'text': 'Bộ lông sặc sỡ của loài vẹt không chỉ để làm đẹp mà là kết quả của quá trình tiến hóa hàng triệu năm nhằm giúp chúng sinh tồn trong môi trường tự nhiên (chủ yếu là các khu rừng nhiệt đới). Dưới đây là các lý do chính:\n\n### 1. Ngụy trang trong môi trường nhiệt đới\nNghe có vẻ mâu thuẫn vì lông vẹt rất chói mắt, nhưng trong tán lá rừng nhiệt đới dày đặc, ánh sáng mặt trời chiếu xuyên qua tạo thành các mảng sáng và bóng tối hỗn độn. \n* Bộ lông màu xanh lá cây, vàng, đỏ hoặc cam của vẹt giúp chúng hòa lẫn vào khung cảnh xung quanh (lá cây, hoa quả, ánh nắng). \n* Khi đậu yên trên cây để ăn quả, những kẻ săn mồi (như chim ăn thịt hoặc khỉ) sẽ rất khó phát hiện ra chúng.\n\n### 2. Thu hút bạn tình (Chọn lọc tự nhiên)\nTrong thế giới loài chim, vẹt cái thường chọn những con vẹt đực có bộ lông rực rỡ và khỏe mạnh nhất làm bạn đời. Bộ lông sặc sỡ là dấu hiệu cho thấy:\n* Con vẹt đó có sức khỏe tốt.\n* Nó có khả năng kiếm ăn giỏi (vì phải ăn đủ chất thì lông

Một danh sách message có thể được truyền vào chat model để đại diện cho lịch sử cuộc hội thoại. Mỗi message có một role mà model sử dụng để xác định ai đã gửi message đó trong cuộc hội thoại.

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)

content=[{'type': 'text', 'text': "J'adore créer des applications.", 'extras': {'signature': 'El4KXAERTTIPk2Q0S297q2GrC4S/Eq/nr80Ry1127MsripSIfSrgiILzfH9h0QSUvAdh/rF9qqGO6d8b9rMo15arl+teQlzKq4R5lvGmZBJP+XNtuUW201mX60UxejTV'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a01d6b-5fcf-7332-bf5a-17525d1dcb7a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 34, 'output_tokens': 7, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}}


In [4]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)

content=[{'type': 'text', 'text': "J'adore créer des applications.", 'extras': {'signature': 'El4KXAERTTIP4qWuf+AZhRbP5HpHhXcr6ZFrr5aXyVdMYScmpY9vI4JN4dyZiC3k+EDAOdPgwf+vtkW0qB8Qt7VWDQe/nkjYCKMaa8qHwFskWuQdL9FCVbjpxbOY+5FW'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a01d6c-be1d-7130-ac5c-f8f74d9b7ac4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 34, 'output_tokens': 7, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}}


### Stream

Hầu hết các model có thể stream nội dung output ngay khi nó được tạo ra. Bằng cách hiển thị output theo tiến trình, streaming cải thiện đáng kể trải nghiệm người dùng, đặc biệt với các phản hồi dài.

Gọi `stream()` trả về một iterator trả ra các chunk output ngay khi chúng được tạo ra. Bạn có thể dùng một loop để xử lý từng chunk theo thời gian thực:

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

for chunk in model.stream("Tại sao vẹt lại có bộ lông sặc sỡ?"):
    print(chunk.text, end="|", flush=True)

Bộ lông sặc sỡ của loài vẹt không chỉ để| làm đẹp mà đóng vai trò rất quan trọng cho sự sinh tồn và phát triển của chúng trong tự nhiên. Dưới đây là| những lý do chính:

**1. Ngụy trang trong môi trường sống (Màu xanh lá và xanh dương|)**
Phần lớn các loài vẹt sống trong các khu rừng nhiệt đới hoặc rừng mưa, nơi có rất| nhiều lá cây và tán lá rậm rạp. 
* Bộ lông màu xanh lá cây hoặc xanh dương giúp| chúng **ngụy trang hoàn hảo** khỏi những kẻ săn mồi (như chim ăn thịt, khỉ hoặc| rắn) khi chúng đậu trên cây. 
* Đối với vẹt, màu sắc sặc sỡ không có nghĩa| là dễ bị phát hiện; trong môi trường nhiệt đới ngập tràn ánh nắng và tán lá xanh, những màu này| lại giúp chúng hòa lẫn vào khung cảnh xung quanh.

**2. Thu hút bạn tình (Giao phối)**
Trong| thế giới loài chim, vẹt là loài có thị giác rất phát triển (chúng có thể nhìn thấy cả tia| cực tím mà mắt người không thấy được). Bộ lông rực rỡ, óng ả là dấu hiệu cho| thấy:
* Sức khỏe tốt.
* Gen tốt.
* Khả năng tìm kiếm thức ăn hiệu quả.
|Những co

Khác với `invoke()`, phương thức trả về một `AIMessage` duy nhất sau khi model hoàn tất việc tạo toàn bộ phản hồi, `stream()` trả về nhiều object `AIMessageChunk`, mỗi object chứa một phần của văn bản output. Điều quan trọng là mỗi chunk trong một stream được thiết kế để có thể gộp lại thành một message hoàn chỉnh thông qua phép cộng:

In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

full = None
for chunk in model.stream("Bầu trời có màu gì?"):
    full = chunk if full is None else full + chunk

print(full.content_blocks)

[{'type': 'text', 'text': 'Bầu trời thường có **màu xanh dương** vào ban ngày. \n\nTuy nhiên, màu sắc của bầu trời có thể thay đổi tùy thuộc vào thời điểm trong ngày và thời tiết:\n*   **Buổi sáng sớm hoặc hoàng hôn:** Có màu đỏ, cam, hồng hoặc vàng.\n*   **Ban đêm:** Có màu đen (hoặc xanh đen).\n*   **Khi trời nhiều mây:** Có màu trắng hoặc xám.\n\n**Giải thích nhanh:** Sở dĩ ban ngày trời màu xanh là do hiện tượng tán xạ ánh sáng Mặt Trời của khí quyển Trái Đất (hiện tượng Rayleigh), trong đó ánh sáng xanh có bước sóng ngắn nên bị tán xạ mạnh hơn các màu khác.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIP1vZfzXh712Yw71WBmB0UijMIsTXlXXXHK4o8pYXAlW19LN5kJPPWqbYH2z4LzO8CRCY5UdrWb8zMendQrUFDo7i19dIYxAczw2HXIoHSHlI1rweuWr8G'}}]


Message thu được có thể được xử lý giống như một message được tạo ra bằng `invoke()` - ví dụ, nó có thể được đưa vào lịch sử message và truyền lại cho model làm context hội thoại.

### Batch

Việc batch một tập các request độc lập đến model có thể cải thiện đáng kể hiệu suất và giảm chi phí, vì việc xử lý có thể được thực hiện song song:

In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

responses = model.batch([
    "Tại sao vẹt lại có bộ lông sặc sỡ?",
    "Máy bay bay như thế nào?",
    "Điện toán lượng tử là gì?"
])
for response in responses:
    print(response)

content=[{'type': 'text', 'text': 'Vẹt có bộ lông sặc sỡ là kết quả của quá trình tiến hóa hàng triệu năm, nhằm phục vụ cho các mục đích sinh tồn quan trọng trong môi trường tự nhiên (chủ yếu là các khu rừng nhiệt đới). Dưới đây là những lý do chính:\n\n**1. Ngụy trang trong môi trường sống (Màu sắc tự nhiên)**\nMặc dù trông có vẻ rất nổi bật với con người, nhưng trong tán lá rừng nhiệt đới rậm rạp – nơi ánh sáng mặt trời chiếu xuống tạo ra những mảng sáng và bóng tối đan xen – bộ lông màu xanh lá, vàng, đỏ hoặc xanh dương của vẹt lại giúp chúng hòa lẫn vào môi trường. Ví dụ, một con vẹt màu xanh lá cây gần như "vô hình" khi đậu trên tán cây xanh để trốn kẻ thù.\n\n**2. Thu hút bạn tình (Chọn lọc giới tính)**\nTrong thế giới loài chim, màu sắc rực rỡ thường là dấu hiệu của sức khỏe tốt, gen trội và khả năng tìm kiếm thức ăn giỏi. Những con vẹt có bộ lông sặc sỡ, bóng mượt sẽ dễ dàng thu hút được bạn tình hơn trong mùa sinh sản. Chim mái thường chọn những con trống có màu sắc nổi bật nh

Theo mặc định, `batch()` chỉ trả về output cuối cùng cho toàn bộ batch. Nếu bạn muốn nhận output cho từng input riêng lẻ ngay khi nó hoàn thành, bạn có thể stream kết quả bằng `batch_as_completed()`:

In [9]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

for response in model.batch_as_completed([
    "Tại sao vẹt lại có bộ lông sặc sỡ?",
    "Máy bay bay như thế nào?",
    "Điện toán lượng tử là gì?"
]):
    print(response)

(0, AIMessage(content=[{'type': 'text', 'text': 'Bộ lông sặc sỡ của loài vẹt không chỉ để làm đẹp mà đóng vai trò rất quan trọng đối với sự sinh tồn và phát triển của chúng trong tự nhiên. Dưới đây là các lý do chính:\n\n### 1. Ngụy trang trong môi trường sống (Cơ chế sinh tồn)\nPhần lớn các loài vẹt sống trong các khu rừng nhiệt đới – nơi có ánh sáng mặt trời chiếu qua vòm lá tạo ra những mảng sáng tối, cùng với vô số loài hoa quả, lá cây đầy màu sắc. \n* Bộ lông xanh lá, đỏ, vàng, cam... của vẹt giúp chúng **hòa lẫn vào tán lá rậm rạp và những chùm quả chín**. \n* Khi đậu im lặng trên cây, những kẻ săn mồi (như chim ăn thịt) sẽ rất khó phát hiện ra chúng.\n\n### 2. Thu hút bạn tình (Giao phối)\nTrong thế giới loài chim, màu sắc lông thường là tiêu chuẩn để đánh giá sức khỏe và vẻ đẹp.\n* Vẹt sử dụng bộ lông sặc sỡ để **thu hút sự chú ý của con khác giới** trong mùa giao phối.\n* Những con vẹt có bộ lông rực rỡ, óng ả thường chứng tỏ chúng khỏe mạnh, có nguồn dinh dưỡng tốt và có bộ g

Khi sử dụng `batch_as_completed()`, kết quả có thể trả về không theo đúng thứ tự. Mỗi kết quả đều bao gồm input index để đối chiếu, giúp khôi phục lại thứ tự ban đầu khi cần.

Khi xử lý số lượng lớn input bằng `batch()` hoặc `batch_as_completed()`, bạn có thể muốn kiểm soát số lệnh gọi song song tối đa. Điều này có thể thực hiện bằng cách thiết lập attribute `max_concurrency` trong dictionary `RunnableConfig`.

```python
model.batch(
    list_of_inputs,
    config={
        'max_concurrency': 5,  # Giới hạn tối đa 5 lệnh gọi song song
    }
)
```

## Tool calling

Model có thể yêu cầu gọi các tool để thực hiện các nhiệm vụ như lấy dữ liệu từ database, tìm kiếm trên web, hoặc chạy code. Tool là sự kết hợp của:

1. Một schema, bao gồm tên của tool, một mô tả, và/hoặc định nghĩa argument (thường là JSON schema)
2. Một function hoặc coroutine để thực thi.

> Bạn có thể nghe thuật ngữ "function calling". Chúng tôi sử dụng thuật ngữ này thay thế cho nhau với "tool calling".

Dưới đây là luồng tool calling cơ bản giữa người dùng và model:

```mermaid theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
sequenceDiagram
    participant U as User
    participant M as Model
    participant T as Tools

    U->>M: "What's the weather in SF and NYC?"
    M->>M: Analyze request & decide tools needed

    par Parallel Tool Calls
        M->>T: get_weather("San Francisco")
        M->>T: get_weather("New York")
    end

    par Tool Execution
        T-->>M: SF weather data
        T-->>M: NYC weather data
    end

    M->>M: Process results & generate response
    M->>U: "SF: 72°F sunny, NYC: 68°F cloudy"
```

Để các tool bạn đã định nghĩa có thể được model sử dụng, bạn phải gắn chúng bằng [`bind_tools`]. Trong các lệnh gọi tiếp theo, model có thể chọn gọi bất kỳ tool nào đã được bind khi cần.

Một số provider model cung cấp các built-in tool (các tool được thực thi ở phía server, như web search và code interpreter) có thể được kích hoạt thông qua các parameter của model hoặc invocation.

In [10]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

@tool
def get_weather(location: str) -> str:
    """Lấy thông tin thời tiết tại một địa điểm."""
    return f"Trời nắng ở {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("Thời tiết ở Boston như thế nào?")
for tool_call in response.tool_calls:
    # Xem các tool call mà model đã tạo
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


Khi gắn các tool do người dùng định nghĩa, phản hồi của model sẽ bao gồm một **yêu cầu** thực thi tool. Khi sử dụng model một cách độc lập (không thông qua agent), bạn cần tự thực thi tool được yêu cầu và trả kết quả về cho model để sử dụng trong các bước reasoning tiếp theo. Khi sử dụng agent, agent loop sẽ tự xử lý toàn bộ vòng lặp thực thi tool cho bạn.

## Structured output

Model có thể được yêu cầu cung cấp phản hồi theo một format khớp với schema đã cho. Điều này hữu ích để đảm bảo output có thể dễ dàng được parse và sử dụng trong các bước xử lý tiếp theo. LangChain hỗ trợ nhiều loại schema và phương pháp để ép buộc structured output.

In [12]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

class Movie(BaseModel):
    """Một bộ phim với các thông tin chi tiết."""
    title: str = Field(description="Tên của bộ phim")
    year: int = Field(description="Năm phát hành của bộ phim")
    director: str = Field(description="Đạo diễn của bộ phim")
    rating: float = Field(description="Điểm đánh giá của bộ phim trên thang 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Cung cấp thông tin chi tiết về bộ phim Inception.")
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


**Những điều cần lưu ý đối với structured output**

* **Parameter method**: Một số provider hỗ trợ các phương pháp khác nhau cho structured output:
    * `'json_schema'`: Sử dụng các tính năng structured output chuyên dụng do provider cung cấp.
    * `'function_calling'`: Tạo ra structured output bằng cách ép buộc một [tool call](#tool-calling) tuân theo schema đã cho.
    * `'json_mode'`: Phương pháp tiền thân của `'json_schema'` được một số provider cung cấp. Tạo ra JSON hợp lệ, nhưng schema phải được mô tả trong prompt.
* **Include raw**: Thiết lập `include_raw=True` để nhận cả output đã parse và raw AI message.
* **Validation**: Pydantic model cung cấp tính năng validation tự động. `TypedDict` và JSON Schema yêu cầu validation thủ công.

Sẽ hữu ích nếu bạn trả về object `AIMessage` gốc cùng với biểu diễn đã parse để truy cập các metadata phản hồi như token count. Để làm điều đó, hãy thiết lập `include_raw=True` khi gọi `with_structured_output`.

In [13]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

class Movie(BaseModel):
    """Một bộ phim với các thông tin chi tiết."""
    title: str = Field(description="Tên của bộ phim")
    year: int = Field(description="Năm phát hành của bộ phim")
    director: str = Field(description="Đạo diễn của bộ phim")
    rating: float = Field(description="Điểm đánh giá của bộ phim trên thang 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Cung cấp thông tin chi tiết về bộ phim Inception.")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8\n}', 'extras': {'signature': 'El4KXAERTTIPPY9V5lb/c06L65ANhQ06GkSyALaxmigU70HQMw2q6oiwKwaZTXlul7HQOO/L7KZjQBqYtxEpja3I1BIKXAYAQgAkOXlOO5qnePVk2kQYkOvutM2khGCG'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01e31-34fd-7781-8b50-7d514b5beda3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 41, 'total_tokens': 55, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

Schema có thể được lồng vào nhau:

In [14]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Kinh phí sản xuất, tính theo triệu USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Cung cấp thông tin chi tiết về bộ phim Inception.")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

## Các chủ đề nâng cao

### Model profile

> Model profile yêu cầu `langchain>=1.1`.

Chat model của LangChain có thể phơi bày một dictionary các tính năng và khả năng được hỗ trợ thông qua attribute `profile`:

```python
model.profile
# {
#   "max_input_tokens": 400000,
#   "image_inputs": True,
#   "reasoning_output": True,
#   "tool_calling": True,
#   ...
# }
```

Phần lớn dữ liệu model profile được cung cấp bởi dự án [models.dev](https://github.com/sst/models.dev), một sáng kiến mã nguồn mở cung cấp dữ liệu về khả năng của các model. Dữ liệu này được bổ sung thêm các field khác cho mục đích sử dụng với LangChain. Các bổ sung này được giữ đồng bộ với dự án gốc khi nó phát triển.

Dữ liệu model profile cho phép ứng dụng thích ứng động với khả năng của model. Ví dụ:

1. [Summarization middleware](/oss/python/langchain/middleware/built-in#summarization) có thể kích hoạt việc tóm tắt dựa trên kích thước context window của model.
2. Chiến lược [structured output](/oss/python/langchain/structured-output) trong `create_agent` có thể được tự động suy luận (ví dụ, bằng cách kiểm tra hỗ trợ cho các tính năng structured output gốc).
3. Model input có thể được kiểm soát dựa trên các [modality](#multimodal) được hỗ trợ và số token input tối đa.
4. [Deep Agents Code](/oss/deepagents/code) lọc [interactive model switcher](/oss/deepagents/code/providers#which-models-appear-in-the-switcher) chỉ hiển thị các model mà profile báo cáo hỗ trợ `tool_calling` và text I/O, đồng thời hiển thị kích thước context window và các cờ (flag) khả năng trong giao diện chi tiết của bộ chọn.

<Accordion title="Cập nhật hoặc ghi đè dữ liệu profile">
  Dữ liệu model profile có thể được thay đổi nếu nó bị thiếu, lỗi thời hoặc không chính xác.

  **Phương án 1 (khắc phục nhanh)**

  Bạn có thể khởi tạo một chat model với bất kỳ profile hợp lệ nào:

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  custom_profile = {
      "max_input_tokens": 100_000,
      "tool_calling": True,
      "structured_output": True,
      # ...
  }
  model = init_chat_model("...", profile=custom_profile)
  ```

  `profile` cũng là một `dict` thông thường và có thể được cập nhật trực tiếp. Nếu model instance được chia sẻ, hãy xem xét sử dụng `model_copy` để tránh làm thay đổi (mutate) trạng thái chia sẻ.

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  new_profile = model.profile | {"key": "value"}
  model.model_copy(update={"profile": new_profile})
  ```

  **Phương án 2 (khắc phục dữ liệu ở nguồn gốc - upstream)**

  Nguồn dữ liệu chính là dự án [models.dev](https://models.dev/). Dữ liệu này được gộp với các field và override bổ sung trong các [package tích hợp](/oss/python/integrations/providers/overview) của LangChain, và được phân phối cùng với các package đó.

  Dữ liệu model profile có thể được cập nhật theo quy trình sau:

  1. (Nếu cần) cập nhật dữ liệu nguồn tại [models.dev](https://models.dev/) thông qua một pull request đến [repository trên GitHub](https://github.com/sst/models.dev).
  2. (Nếu cần) cập nhật các field và override bổ sung trong `langchain_<package>/data/profile_augmentations.toml` thông qua một pull request đến [package tích hợp](/oss/python/integrations/providers/overview) của LangChain.
  3. Sử dụng CLI tool [`langchain-model-profiles`](https://pypi.org/project/langchain-model-profiles/) để lấy dữ liệu mới nhất từ [models.dev](https://models.dev/), gộp các bổ sung và cập nhật dữ liệu profile:

  <CodeGroup>
    ```bash pip theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    pip install -U langchain-model-profiles
    ```

    ```bash uv theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    uv add langchain-model-profiles
    ```
  </CodeGroup>

  ```bash theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  langchain-profiles refresh --provider <provider> --data-dir <data_dir>
  ```

  Lệnh này sẽ:

  * Tải xuống dữ liệu mới nhất cho `<provider>` từ models.dev
  * Gộp các bổ sung từ `profile_augmentations.toml` trong `<data_dir>`
  * Ghi các profile đã gộp vào `profiles.py` trong `<data_dir>`

  Ví dụ: từ [`libs/partners/anthropic`](https://github.com/langchain-ai/langchain/tree/master/libs/partners/anthropic) trong [LangChain monorepo](https://github.com/langchain-ai/langchain):

  ```bash theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  uv run --with langchain-model-profiles --provider anthropic --data-dir langchain_anthropic/data
  ```
</Accordion>

<Warning>
  Model profile là một tính năng beta. Format của một profile có thể thay đổi.
</Warning>

### Multimodal

Một số model có khả năng xử lý và trả về dữ liệu không phải văn bản như hình ảnh, âm thanh và video. Bạn có thể truyền dữ liệu không phải văn bản vào model bằng cách cung cấp các [content block](/oss/python/langchain/messages#message-content).

<Tip>
  Tất cả chat model của LangChain có khả năng multimodal đều hỗ trợ:

  1. Dữ liệu theo format chuẩn liên provider (cross-provider standard format) (xem [hướng dẫn về message](/oss/python/langchain/messages) của chúng tôi)
  2. Format OpenAI [chat completions](https://platform.openai.com/docs/api-reference/chat)
  3. Bất kỳ format nào là gốc (native) của provider cụ thể đó (ví dụ, model của Anthropic nhận format gốc của Anthropic)
</Tip>

Xem [phần multimodal](/oss/python/langchain/messages#multimodal) trong hướng dẫn message để biết chi tiết.

<Tooltip tip="Không phải mọi LLM đều được tạo ra như nhau!" cta="Xem tài liệu tham khảo" href="https://models.dev/">Một số model</Tooltip> có thể trả về dữ liệu multimodal như một phần của phản hồi. Nếu được gọi để làm vậy, [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) thu được sẽ có các content block với type multimodal.

```python Multimodal output theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
response = model.invoke("Create a picture of a cat")
print(response.content_blocks)
# [
#     {"type": "text", "text": "Here's a picture of a cat"},
#     {"type": "image", "base64": "...", "mime_type": "image/jpeg"},
# ]
```

Xem [trang tích hợp](/oss/python/integrations/providers/overview) để biết chi tiết về các provider cụ thể.

### Reasoning

Nhiều model có khả năng thực hiện reasoning nhiều bước để đi đến kết luận. Điều này bao gồm việc chia nhỏ các vấn đề phức tạp thành các bước nhỏ hơn, dễ quản lý hơn.

**Nếu được model gốc hỗ trợ,** bạn có thể hiển thị quá trình reasoning này để hiểu rõ hơn cách model đưa ra câu trả lời cuối cùng.

<CodeGroup>
  ```python Stream reasoning output theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  for chunk in model.stream("Why do parrots have colorful feathers?"):
      reasoning_steps = [r for r in chunk.content_blocks if r["type"] == "reasoning"]
      print(reasoning_steps if reasoning_steps else chunk.text)
  ```

  ```python Complete reasoning output theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  response = model.invoke("Why do parrots have colorful feathers?")
  reasoning_steps = [b for b in response.content_blocks if b["type"] == "reasoning"]
  print(" ".join(step["reasoning"] for step in reasoning_steps))
  ```
</CodeGroup>

Tùy vào model, đôi khi bạn có thể chỉ định mức độ "effort" (công sức) mà model nên đầu tư cho reasoning. Tương tự, bạn có thể yêu cầu model tắt hoàn toàn reasoning. Điều này có thể ở dạng các "tier" (cấp độ) phân loại của reasoning (ví dụ `'low'` hoặc `'high'`) hoặc dạng số nguyên token budget.

<Note>
  `reasoning_effort` như một parameter chuẩn yêu cầu `langchain-core>=1.5.2`, cùng với phiên bản package partner tương ứng: `langchain-anthropic>=1.5.3`, `langchain-openai>=1.4.1`, `langchain-fireworks>=1.5.2`, `langchain-xai>=1.3.0`, `langchain-google-genai>=4.3.1`, hoặc `langchain-aws>=1.6.5`.
</Note>

[`ChatOpenAI`](https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI), [`ChatAnthropic`](https://reference.langchain.com/python/langchain-anthropic/chat_models/ChatAnthropic), [`ChatFireworks`](https://reference.langchain.com/python/langchain-fireworks/chat_models/ChatFireworks), [`ChatXAI`](https://reference.langchain.com/python/langchain-xai/chat_models/ChatXAI), [`ChatGoogleGenerativeAI`](https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI), và [`ChatBedrockConverse`](https://reference.langchain.com/python/langchain-aws/chat_models/bedrock_converse/ChatBedrockConverse) đều hỗ trợ parameter chuẩn `reasoning_effort`. Giống như `temperature`, nó có thể được thiết lập khi tạo model hoặc theo từng lần invoke, và mỗi provider sẽ chuyển đổi nó sang format API riêng của mình:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6")
response = model.invoke(
    "Why do parrots have colorful feathers?",
    reasoning_effort="high",
)
```

Các mức effort được hỗ trợ và giá trị mặc định theo tài liệu của provider sẽ khác nhau tùy theo model. Hãy kiểm tra [profile](#model-profiles) của model để biết các mức được hỗ trợ và mức mặc định của nó:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
model.profile["reasoning_effort_levels"]  # ví dụ: ['low', 'medium', 'high']
model.profile["reasoning_effort_default"]  # ví dụ: 'high'
```

Một số provider cũng chấp nhận một tên gọi khác (native alias) cho `reasoning_effort` (ví dụ, `ChatAnthropic` chấp nhận `effort` và `ChatGoogleGenerativeAI` chấp nhận `thinking_level`). Xem trang [chat model integrations](/oss/python/integrations/chat) để biết chi tiết theo từng provider.

Để biết chi tiết, xem [trang tích hợp](/oss/python/integrations/providers/overview) hoặc [tài liệu tham khảo](https://reference.langchain.com/python/integrations/) cho chat model tương ứng của bạn.

### Model chạy local

LangChain hỗ trợ chạy model cục bộ (local) trên phần cứng của riêng bạn. Điều này hữu ích cho các trường hợp khi bảo mật dữ liệu là yếu tố then chốt, khi bạn muốn gọi một model tùy chỉnh, hoặc khi bạn muốn tránh chi phí phát sinh khi sử dụng model dựa trên cloud.

[Ollama](/oss/python/integrations/chat/ollama) là một trong những cách dễ nhất để chạy chat model và embedding model cục bộ.

### Prompt caching

Nhiều provider cung cấp tính năng prompt caching để giảm độ trễ và chi phí khi xử lý lại các token giống nhau. Bạn có thể sử dụng caching ở ba cấp độ:

* **Implicit provider caching (caching ngầm định của provider):** provider tự động chuyển tiếp phần tiết kiệm chi phí nếu một request trúng cache, không cần cấu hình gì thêm. Ví dụ: [OpenAI](/oss/python/integrations/chat/openai) và [Gemini](/oss/python/integrations/chat/google_generative_ai).
* **Provider-level explicit controls (kiểm soát rõ ràng ở cấp provider):** provider cho phép bạn chỉ định thủ công các cache point để kiểm soát tốt hơn hoặc đảm bảo tiết kiệm chi phí. Các control này phản ánh đúng hành vi provider/API gốc. Ví dụ:
  * [`ChatOpenAI`](https://reference.langchain.com/python/langchain-openai/chat_models/base/ChatOpenAI) (qua `prompt_cache_key`)
  * Anthropic content-block [`cache_control`](/oss/python/integrations/chat/anthropic#prompt-caching)
  * [Gemini](https://reference.langchain.com/python/integrations/langchain_google_genai/).
  * Các block [`cachePoint`](/oss/python/integrations/chat/bedrock#prompt-caching) của AWS Bedrock
* **LangChain middleware:** đối với agent, middleware cho phép LangChain tối ưu hóa việc caching cho system prompt và nội dung tool ổn định. Ví dụ:
  * [`AnthropicPromptCachingMiddleware`](/oss/python/integrations/middleware/anthropic#prompt-caching) của Anthropic
  * [`BedrockPromptCachingMiddleware`](/oss/python/integrations/middleware/aws#prompt-caching) của AWS Bedrock

<Warning>
  Prompt caching thường chỉ được kích hoạt khi vượt ngưỡng số token input tối thiểu. Xem [trang của provider](/oss/python/integrations/chat) để biết chi tiết.
</Warning>

Việc sử dụng cache sẽ được phản ánh trong [usage metadata](/oss/python/langchain/messages#token-usage) của phản hồi model.

### Sử dụng tool ở phía server (Server-side tool use)

Một số provider hỗ trợ các vòng lặp [tool-calling](#tool-calling) ở phía server: model có thể tương tác với web search, code interpreter và các tool khác, đồng thời phân tích kết quả trong một lượt hội thoại (turn) duy nhất.

Nếu model gọi một tool ở phía server, nội dung của message phản hồi sẽ bao gồm nội dung đại diện cho lệnh gọi và kết quả của tool. Truy cập [content block](/oss/python/langchain/messages#standard-content-blocks) của phản hồi sẽ trả về các server-side tool call và kết quả theo một format không phụ thuộc provider (provider-agnostic):

```python Invoke with server-side tool use theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.4-mini")

tool = {"type": "web_search"}
model_with_tools = model.bind_tools([tool])

response = model_with_tools.invoke("What was a positive news story from today?")
print(response.content_blocks)
```

```python Result expandable theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
[
    {
        "type": "server_tool_call",
        "name": "web_search",
        "args": {
            "query": "positive news stories today",
            "type": "search"
        },
        "id": "ws_abc123"
    },
    {
        "type": "server_tool_result",
        "tool_call_id": "ws_abc123",
        "status": "success"
    },
    {
        "type": "text",
        "text": "Here are some positive news stories from today...",
        "annotations": [
            {
                "end_index": 410,
                "start_index": 337,
                "title": "article title",
                "type": "citation",
                "url": "..."
            }
        ]
    }
]
```

Đây đại diện cho một lượt hội thoại (turn) duy nhất; không có object [ToolMessage](/oss/python/langchain/messages#tool-message) tương ứng nào cần phải truyền vào giống như trong [tool-calling](#tool-calling) ở phía client.

Xem [trang tích hợp](/oss/python/integrations/chat) cho provider tương ứng của bạn để biết các tool có sẵn và chi tiết sử dụng.

### Rate limiting

Nhiều provider chat model áp đặt giới hạn về số lượng lệnh gọi có thể thực hiện trong một khoảng thời gian nhất định. Nếu bạn chạm giới hạn (rate limit), bạn thường sẽ nhận được một phản hồi lỗi rate limit từ provider, và cần chờ trước khi thực hiện thêm request.

Để giúp quản lý rate limit, các tích hợp chat model chấp nhận một parameter `rate_limiter`, có thể được cung cấp khi khởi tạo để kiểm soát tốc độ gửi request.

<Accordion title="Khởi tạo và sử dụng rate limiter" icon="gauge">
  LangChain đi kèm với (một tính năng tùy chọn) [`InMemoryRateLimiter`](https://reference.langchain.com/python/langchain-core/rate_limiters/InMemoryRateLimiter) tích hợp sẵn. Limiter này an toàn với thread (thread-safe) và có thể được chia sẻ giữa nhiều thread trong cùng một process.

  ```python Define a rate limiter theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  from langchain.rate_limiters import InMemoryRateLimiter

  rate_limiter = InMemoryRateLimiter(
      requests_per_second=0.1,  # 1 request mỗi 10 giây
      check_every_n_seconds=0.1,  # Kiểm tra mỗi 100ms xem có được phép gửi request không
      max_bucket_size=10,  # Kiểm soát kích thước burst tối đa.
  )

  model = init_chat_model(
      model="gpt-5.5",
      model_provider="openai",
      rate_limiter=rate_limiter  # [!code highlight]
  )
  ```

  <Warning>
    Rate limiter được cung cấp chỉ có thể giới hạn số lượng request theo đơn vị thời gian. Nó sẽ không giúp được gì nếu bạn cần giới hạn dựa trên kích thước của request.
  </Warning>
</Accordion>

### Cấu hình Base URL và proxy

Bạn có thể cấu hình một base URL tùy chỉnh cho các provider triển khai OpenAI Chat Completions API.

<Warning>
  `model_provider="openai"` (hoặc sử dụng trực tiếp `ChatOpenAI`) hướng đến đặc tả OpenAI API chính thức. Các field đặc thù của provider từ các router và proxy có thể không được trích xuất hoặc giữ lại.

  Đối với OpenRouter và LiteLLM, hãy ưu tiên sử dụng các tích hợp chuyên dụng:

  * [OpenRouter qua `ChatOpenRouter`](/oss/python/integrations/chat/openrouter) (`langchain-openrouter`)
  * [LiteLLM qua `ChatLiteLLM` / `ChatLiteLLMRouter`](/oss/python/integrations/chat) (`langchain-litellm`)
</Warning>

<Accordion title="Base URL tùy chỉnh" icon="link">
  Nhiều provider model cung cấp API tương thích OpenAI (ví dụ, [Together AI](https://www.together.ai/), [vLLM](https://github.com/vllm-project/vllm)). Bạn có thể sử dụng [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) với các provider này bằng cách chỉ định parameter `base_url` phù hợp:

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  model = init_chat_model(
      model="MODEL_NAME",
      model_provider="openai",
      base_url="BASE_URL",
      api_key="YOUR_API_KEY",
  )
  ```

  <Note>
    Khi khởi tạo trực tiếp bằng chat model class, tên parameter có thể khác nhau tùy theo provider. Xem [tài liệu tham khảo](/oss/python/integrations/providers/overview) tương ứng để biết chi tiết.
  </Note>
</Accordion>

<Accordion title="Cấu hình HTTP proxy" icon="shield">
  Đối với các triển khai (deployment) yêu cầu HTTP proxy, một số tích hợp model hỗ trợ cấu hình proxy:

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  from langchain_openai import ChatOpenAI

  model = ChatOpenAI(
      model="gpt-5.5",
      openai_proxy="http://proxy.example.com:8080"
  )
  ```

  <Note>
    Việc hỗ trợ proxy khác nhau tùy theo từng tích hợp. Xem [tài liệu tham khảo](/oss/python/integrations/providers/overview) của provider model cụ thể để biết các tùy chọn cấu hình proxy.
  </Note>
</Accordion>

### Log probability

Một số model có thể được cấu hình để trả về log probability ở cấp độ token, đại diện cho khả năng xuất hiện của một token cho trước, bằng cách thiết lập parameter `logprobs` khi khởi tạo model:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
model = init_chat_model(
    model="gpt-5.5",
    model_provider="openai"
).bind(logprobs=True)

response = model.invoke("Why do parrots talk?")
print(response.response_metadata["logprobs"])
```

### Token usage

Nhiều provider model trả về thông tin sử dụng token như một phần của phản hồi invoke. Khi có sẵn, thông tin này sẽ được bao gồm trong các object [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) được tạo ra bởi model tương ứng. Để biết thêm chi tiết, xem hướng dẫn [messages](/oss/python/langchain/messages).

<Note>
  Một số API của provider, đặc biệt là OpenAI và Azure OpenAI chat completions, yêu cầu người dùng phải chủ động opt-in để nhận dữ liệu sử dụng token trong ngữ cảnh streaming. Xem phần [streaming usage metadata](/oss/python/integrations/chat/openai#streaming-usage-metadata) trong hướng dẫn tích hợp để biết chi tiết.
</Note>

Bạn có thể theo dõi tổng số lượng token sử dụng trên nhiều model trong một ứng dụng bằng cách sử dụng callback hoặc context manager, như minh họa dưới đây:

<Tabs>
  <Tab title="Callback handler">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.chat_models import init_chat_model
    from langchain_core.callbacks import UsageMetadataCallbackHandler

    model_1 = init_chat_model(model="gpt-5.4-mini")
    model_2 = init_chat_model(model="claude-haiku-4-5-20251001")

    callback = UsageMetadataCallbackHandler()
    result_1 = model_1.invoke("Hello", config={"callbacks": [callback]})
    result_2 = model_2.invoke("Hello", config={"callbacks": [callback]})
    print(callback.usage_metadata)
    ```

    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    {
        'gpt-5.4-mini': {
            'input_tokens': 8,
            'output_tokens': 10,
            'total_tokens': 18,
            'input_token_details': {'audio': 0, 'cache_read': 0},
            'output_token_details': {'audio': 0, 'reasoning': 0}
        },
        'claude-haiku-4-5-20251001': {
            'input_tokens': 8,
            'output_tokens': 21,
            'total_tokens': 29,
            'input_token_details': {'cache_read': 0, 'cache_creation': 0}
        }
    }
    ```
  </Tab>
</Tabs>

### Cấu hình invocation

Khi gọi (invoke) một model, bạn có thể truyền thêm cấu hình thông qua parameter `config` bằng dictionary [`RunnableConfig`](https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig). Điều này cho phép kiểm soát runtime đối với hành vi thực thi, callback và theo dõi metadata.

Các tùy chọn cấu hình phổ biến bao gồm:

```python Invocation with config theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
response = model.invoke(
    "Tell me a joke",
    config={
        "run_name": "joke_generation",      # Tên tùy chỉnh cho lần chạy này
        "tags": ["humor", "demo"],          # Tag để phân loại
        "metadata": {"user_id": "123"},     # Metadata tùy chỉnh
        "callbacks": [my_callback_handler], # Các callback handler
    }
)
```

Các giá trị cấu hình này đặc biệt hữu ích khi:

* Debug với tracing của [LangSmith](/langsmith/observability)
* Triển khai logging hoặc giám sát (monitoring) tùy chỉnh
* Kiểm soát việc sử dụng resource trong production
* Theo dõi các lệnh invoke trên các pipeline phức tạp

<Accordion title="Các attribute cấu hình chính">
  <ParamField body="run_name" type="string">
    Định danh cho lệnh invoke cụ thể này trong log và trace. Không được kế thừa bởi các sub-call.
  </ParamField>

  <ParamField body="tags" type="string[]">
    Nhãn được kế thừa bởi tất cả các sub-call để filter và tổ chức trong công cụ debug.
  </ParamField>

  <ParamField body="metadata" type="object">
    Các cặp key-value tùy chỉnh để theo dõi thêm context, được kế thừa bởi tất cả các sub-call.
  </ParamField>

  <ParamField body="max_concurrency" type="number">
    Kiểm soát số lượng lệnh gọi song song tối đa khi sử dụng [`batch()`](https://reference.langchain.com/python/langchain_core/language_models/#langchain_core.language_models.chat_models.BaseChatModel.batch) hoặc [`batch_as_completed()`](https://reference.langchain.com/python/langchain_core/language_models/#langchain_core.language_models.chat_models.BaseChatModel.batch_as_completed).
  </ParamField>

  <ParamField body="callbacks" type="array">
    Các handler để giám sát và phản hồi lại các event trong quá trình thực thi.
  </ParamField>

  <ParamField body="recursion_limit" type="number">
    Độ sâu recursion tối đa cho các chain, để tránh vòng lặp vô hạn trong các pipeline phức tạp.
  </ParamField>
</Accordion>

<Tip>
  Xem đầy đủ tài liệu tham khảo [`RunnableConfig`](https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig) để biết toàn bộ các attribute được hỗ trợ.
</Tip>

### Configurable model

Bạn cũng có thể tạo một model có thể cấu hình được tại runtime bằng cách chỉ định [`configurable_fields`](https://reference.langchain.com/python/langchain_core/language_models/#langchain_core.language_models.chat_models.BaseChatModel.configurable_fields). Nếu bạn không chỉ định giá trị model, thì `'model'` và `'model_provider'` sẽ có thể cấu hình được theo mặc định.

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.chat_models import init_chat_model

configurable_model = init_chat_model(temperature=0)

configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "gpt-5-nano"}},  # Chạy với GPT-5-Nano
)
configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "claude-sonnet-4-6"}},  # Chạy với Claude
)
```

<Accordion title="Configurable model với các giá trị mặc định">
  Chúng ta có thể tạo một configurable model với các giá trị model mặc định, chỉ định parameter nào có thể cấu hình được, và thêm tiền tố (prefix) cho các parameter có thể cấu hình:

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  first_model = init_chat_model(
          model="gpt-5.4-mini",
          temperature=0,
          configurable_fields=("model", "model_provider", "temperature", "max_tokens"),
          config_prefix="first",  # Hữu ích khi bạn có một chain với nhiều model
  )

  first_model.invoke("what's your name")
  ```

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  first_model.invoke(
      "what's your name",
      config={
          "configurable": {
              "first_model": "claude-sonnet-4-6",
              "first_temperature": 0.5,
              "first_max_tokens": 100,
          }
      },
  )
  ```

  Xem tài liệu tham khảo [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) để biết thêm chi tiết về `configurable_fields` và `config_prefix`.
</Accordion>

<Accordion title="Sử dụng configurable model theo cách khai báo (declaratively)">
  Chúng ta có thể gọi các phép toán khai báo như `bind_tools`, `with_structured_output`, `with_configurable`, v.v. trên một configurable model, và chain một configurable model theo cùng cách chúng ta làm với một chat model object được khởi tạo thông thường.

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  from pydantic import BaseModel, Field


  class GetWeather(BaseModel):
      """Lấy thông tin thời tiết hiện tại tại một địa điểm cho trước"""

          location: str = Field(description="Thành phố và tiểu bang, ví dụ: San Francisco, CA")


  class GetPopulation(BaseModel):
      """Lấy dân số hiện tại tại một địa điểm cho trước"""

          location: str = Field(description="Thành phố và tiểu bang, ví dụ: San Francisco, CA")


  model = init_chat_model(temperature=0)
  model_with_tools = model.bind_tools([GetWeather, GetPopulation])

  model_with_tools.invoke(
      "what's bigger in 2024 LA or NYC", config={"configurable": {"model": "gpt-5.4-mini"}}
  ).tool_calls
  ```

  ```
  [
      {
          'name': 'GetPopulation',
          'args': {'location': 'Los Angeles, CA'},
          'id': 'call_Ga9m8FAArIyEjItHmztPYA22',
          'type': 'tool_call'
      },
      {
          'name': 'GetPopulation',
          'args': {'location': 'New York, NY'},
          'id': 'call_jh2dEvBaAHRaw5JUDthOs7rt',
          'type': 'tool_call'
      }
  ]
  ```

  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  model_with_tools.invoke(
      "what's bigger in 2024 LA or NYC",
      config={"configurable": {"model": "claude-sonnet-4-6"}},
  ).tool_calls
  ```

  ```
  [
      {
          'name': 'GetPopulation',
          'args': {'location': 'Los Angeles, CA'},
          'id': 'toolu_01JMufPf4F4t2zLj7miFeqXp',
          'type': 'tool_call'
      },
      {
          'name': 'GetPopulation',
          'args': {'location': 'New York City, NY'},
          'id': 'toolu_01RQBHcE8kEEbYTuuS8WqY1u',
          'type': 'tool_call'
      }
  ]
  ```
</Accordion>

### Chọn model động (Dynamic model selection)

Dynamic model được chọn tại <Tooltip tip="Môi trường thực thi của agent, chứa cấu hình bất biến và dữ liệu ngữ cảnh tồn tại xuyên suốt quá trình thực thi của agent (ví dụ: user ID, session detail, hoặc cấu hình đặc thù của ứng dụng).">runtime</Tooltip> dựa trên <Tooltip tip="Dữ liệu luân chuyển qua quá trình thực thi của agent, bao gồm message, các field tùy chỉnh, và bất kỳ thông tin nào cần được theo dõi và có thể được sửa đổi trong quá trình xử lý (ví dụ: tùy chọn người dùng hoặc số liệu sử dụng tool).">state</Tooltip> và context hiện tại. Điều này cho phép logic routing phức tạp và tối ưu hóa chi phí.

Để sử dụng dynamic model, hãy tạo middleware bằng decorator [`@wrap_model_call`](https://reference.langchain.com/python/langchain/agents/middleware/types/wrap_model_call) để sửa đổi model trong request:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOpenAI(model="gpt-5.4-mini")
advanced_model = ChatOpenAI(model="gpt-5.5")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Chọn model dựa trên độ phức tạp của cuộc hội thoại."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Sử dụng model cao cấp cho các cuộc hội thoại dài hơn
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Model mặc định
    tools=tools,
    middleware=[dynamic_model_selection]
)
```

<Warning>
  Các model đã được pre-bound (model đã được gọi [`bind_tools`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/bind_tools) sẵn) không được hỗ trợ khi sử dụng structured output. Nếu bạn cần chọn model động kết hợp với structured output, hãy đảm bảo các model được truyền vào middleware chưa được pre-bound.
</Warning>

<Tip>
  Để biết chi tiết cấu hình model, xem [Models](/oss/python/langchain/models). Để biết các pattern chọn model động, xem [Dynamic model in middleware](/oss/python/langchain/middleware#dynamic-model).
</Tip>

***

<div className="source-links">
  <Callout icon="terminal-2">
    [Kết nối các tài liệu này](/use-these-docs) với Claude, VSCode, và nhiều công cụ khác qua MCP để nhận câu trả lời theo thời gian thực.
  </Callout>

  <Callout icon="edit">
    [Chỉnh sửa trang này trên GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/models.mdx) hoặc [báo lỗi (file an issue)](https://github.com/langchain-ai/docs/issues/new/choose).
  </Callout>
</div>